# NSB observational validation

NSB models the night-sky background for ground-based astronomy. This executable
experiment compares **NSB Automatic (Oxygen default)**, **explicit continuum-only**, pinned **nsb2**, and
a calibration-only baseline against the same Patat (2008) FORS1 spectra, integrated
over **365–650 nm** in photons cm⁻² ns⁻¹ sr⁻¹. It measures errors, uncertainty,
improvements and regressions under one frozen scientific protocol.

**v0.2.0 Automatic Airglow is PALACE Oxygen** (maintainer release-policy default). This is **not** an independent absolute observational admission; LVM absolute equivalence remains pending. **Patat observations influenced Oxygen development.** These are development data,
including the historically named evaluation partition, not an untouched holdout.
Oxygen remains experimental; this experiment grants no independent scientific admission.

**Single-file use:** download this notebook, open it with a CPython 3.12 or 3.13
Jupyter kernel, edit the configuration below, and Run All. The setup cell installs
public dependencies when enabled. No repository, helper package or private service
is used. Tested on Linux x86-64; other platforms require compatible binary wheels.
Google Colab has not been tested.

**Runtime requirement:** the comparison requires the v0.2.0 public Oxygen
and restricted-band APIs. Published mode requires an actual v0.2.0 PyPI wheel;
before publication, use developer mode with a verified wheel and exact commit
and file SHA-256. A synthetic run is a pipeline test, never evidence of
observational model accuracy.

Code: AGPL-3.0-only, Copyright (C) 2026 Vallés Puig, Ramon. Data retain source terms.


## 1. Select a runtime and set up

`MODE="observational"` is the scientific workflow; `"synthetic"` is a network-free
pipeline fixture. `RUNTIME="published"` downloads an exact PyPI binary wheel;
`"developer"` verifies an already installed wheel against its expected SHA-256.
Start a fresh kernel when changing runtimes. No source compilation occurs here.
`INSTALL_DEPENDENCIES=False` uses a pre-provisioned environment without installation.

For developer mode, separately build/install a public NSB revision before starting
Jupyter. Use an exact source revision implementing the v0.2.0 public API. Record
its full commit SHA; the notebook checks that the required API is present.
For example, in a Python 3.13 virtual environment with Rust and Git installed:

```bash
git clone https://github.com/VPRamon/NSB.git nsb-source
git -C nsb-source checkout --detach <reviewed-full-commit-SHA>
python -m pip install maturin==1.15.0
(cd nsb-source && python -m maturin build --release --locked --interpreter "$(which python)" --out ../wheels)
python -m pip install wheels/*.whl
sha256sum wheels/*.whl
```

Set `NSB_WHEEL`, `NSB_WHEEL_SHA256`, `NSB_COMMIT` and `NSB_VERSION` below to those
actual artifacts. The commit is a **user-supplied build provenance assertion**;
the wheel and imported binary are independently checked. Retain the build log,
clean source status and Cargo.lock with the wheel for stronger source provenance.
Notebook users need only the installed public runtime and wheel, not the checkout.
The notebook installs pinned nsb2 from its public Git revision (Git is required
for this installation). Scientific caches are validated on every use.

If available, `BUILD_RECEIPT` can point to an existing clean-source wheel receipt;
its commit, version, wheel digest and file inventory are checked and retained with
Cargo.lock, compiler/provider versions and other recorded build metadata. It is optional.


In [ ]:
# SPDX-License-Identifier: AGPL-3.0-only
# Copyright (C) 2026 Vallés Puig, Ramon
import os
from pathlib import Path

MODE = os.environ.get("NSB_VALIDATION_MODE", "observational")
RUNTIME = os.environ.get("NSB_VALIDATION_RUNTIME", "published")
NSB_VERSION = os.environ.get("NSB_VALIDATION_VERSION", "0.2.0")
NSB_WHEEL = os.environ.get("NSB_VALIDATION_WHEEL", "")
NSB_WHEEL_SHA256 = os.environ.get("NSB_VALIDATION_WHEEL_SHA256", "")
BUILD_RECEIPT = os.environ.get(
    "NSB_VALIDATION_BUILD_RECEIPT", ""
)  # optional existing build receipt
NSB_COMMIT = os.environ.get("NSB_VALIDATION_COMMIT", "")
INSTALL_DEPENDENCIES = os.environ.get("NSB_VALIDATION_INSTALL", "1") == "1"
OFFLINE_DATA = os.environ.get("NSB_VALIDATION_OFFLINE_DATA", "0") == "1"
CACHE = Path(os.environ.get("NSB_VALIDATION_CACHE", str(Path.home() / ".cache/nsb-observational")))
SOLAR_FILE = os.environ.get("NSB_VALIDATION_SOLAR", "")  # optional registered frozen input
OUTPUT = os.environ.get("NSB_VALIDATION_OUTPUT", "validation-results")  # empty disables export
PREVIOUS_RUN = os.environ.get("NSB_VALIDATION_PREVIOUS", "")  # optional result JSON

# Frozen protocol: changing these creates an incompatible experiment.
REPETITIONS, SEED, PRACTICAL_MARGIN = 2000, 182, 0.10
if MODE not in ("observational", "synthetic") or RUNTIME not in ("published", "developer"):
    raise ValueError("Choose observational/synthetic and published/developer explicitly")

In [ ]:
import importlib
import importlib.metadata as metadata
import subprocess
import sys

# Deliberately separate environment preparation from scientific analysis.
DEPENDENCIES = {
    "numpy": "2.5.4",
    "astropy": "8.0.2",
    "matplotlib": "3.11.2",
    "siderust": "0.5.1",
    "tempoch": "0.5.0",
}
NSB2_COMMIT = "bc9320db03fbff69997ce366c6e76a37339c5d00"


def pip_install(*requirements):
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", *requirements],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    if result.returncode:
        raise RuntimeError("Public dependency installation failed:\n" + result.stdout[-5000:])
    importlib.invalidate_caches()


if not (3, 12) <= sys.version_info[:2] <= (3, 13):
    raise RuntimeError("Use the tested CPython 3.12/3.13 environment")
if MODE == "observational" and INSTALL_DEPENDENCIES:
    missing = []
    for name, version in DEPENDENCIES.items():
        try:
            installed = metadata.version(name)
        except metadata.PackageNotFoundError:
            installed = None
        if installed != version:
            if name in sys.modules:
                raise RuntimeError(
                    f"{name} is already imported; prepare dependencies then restart kernel"
                )
            missing.append(f"{name}=={version}")
    if missing:
        pip_install("--only-binary=:all:", *missing)
    try:
        direct = metadata.distribution("nsb2").read_text("direct_url.json") or ""
    except metadata.PackageNotFoundError:
        direct = ""
    if NSB2_COMMIT not in direct:
        if "nsb2" in sys.modules:
            raise RuntimeError("Restart kernel before installing pinned nsb2")
        pip_install(f"nsb2 @ git+https://github.com/GerritRo/nsb2.git@{NSB2_COMMIT}")

In [ ]:
import ast
import hashlib
import json
import platform
import re
import tarfile
import tempfile
import zipfile
from datetime import UTC, date, datetime, timedelta
from itertools import combinations
from types import SimpleNamespace
from urllib.request import urlopen

import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np
from astropy.constants import c, h
from astropy.coordinates import AltAz, EarthLocation, SkyCoord
from astropy.io import fits
from astropy.time import Time
from astropy.utils import iers
from astropy.utils.data import cache_contents
from IPython import get_ipython
from IPython.display import Markdown, display

iers.conf.auto_download = False
plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
LABEL = (
    "SYNTHETIC — no observational evidence"
    if MODE == "synthetic"
    else "Patat development experiment"
)
display(Markdown(f"**{LABEL}**"))

In [ ]:
def sha256(path):
    result = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            result.update(block)
    return result.hexdigest()


def wheel_files(wheel):
    with zipfile.ZipFile(wheel) as archive:
        return {
            p: hashlib.sha256(archive.read(p)).hexdigest()
            for p in archive.namelist()
            if not p.endswith("/") and ".dist-info/" not in p
        }


def capabilities(nsb):
    required = ("PARANAL_PALACE_V1", "PARANAL_PALACE_OXYGEN_V1")
    if (
        not hasattr(nsb, "AirglowModel")
        or any(not hasattr(nsb.AirglowModel, name) for name in required)
        or not hasattr(nsb.NsbEvaluator, "integrate_band_components")
    ):
        raise RuntimeError(
            "Installed nsb-py lacks required Oxygen/restricted-band API. "
            "Select developer mode with an Oxygen-capable wheel prepared in section 1, or choose a future compatible PyPI release. No model is substituted."
        )


def verify_nsb2():
    """PEP 610 VCS pin plus installed RECORD hashes and imported-path check."""
    import base64
    import csv
    import io

    import nsb2

    dist = metadata.distribution("nsb2")
    direct = json.loads(dist.read_text("direct_url.json") or "{}")
    if (
        direct.get("vcs_info", {}).get("commit_id") != NSB2_COMMIT
        or direct.get("url") != "https://github.com/GerritRo/nsb2.git"
    ):
        raise ValueError("nsb2 must be installed from the registered HTTPS Git revision")
    if Path(nsb2.__file__).resolve() != Path(dist.locate_file("nsb2/__init__.py")).resolve():
        raise ValueError("nsb2 import shadowed by another module")
    hashes = {}
    for name, digest, _size in csv.reader(io.StringIO(dist.read_text("RECORD"))):
        if name.startswith("nsb2/") and digest:
            algorithm, encoded = digest.split("=", 1)
            if algorithm != "sha256":
                raise ValueError("Unsupported nsb2 RECORD hash")
            expected = base64.urlsafe_b64decode(encoded + "==").hex()
            if sha256(dist.locate_file(name)) != expected:
                raise ValueError(f"Modified nsb2 installation: {name}")
            hashes[name] = expected
    if not hashes:
        raise ValueError("No verifiable nsb2 package records")
    return {"commit": NSB2_COMMIT, "version": dist.version, "direct_url": direct, "files": hashes}


def verify_wheel(wheel, digest, version):
    """Verify wheel bytes, every payload file, actual import and loaded native extension."""
    if sha256(wheel) != digest:
        raise ValueError("NSB wheel SHA-256 mismatch")
    dist = metadata.distribution("nsb-py")
    if dist.version != version:
        raise ValueError("Installed NSB version differs from selected version")
    expected = wheel_files(wheel)
    paths = {Path(dist.locate_file(name)).resolve() for name in expected}
    for name, expected_sha in expected.items():
        if sha256(dist.locate_file(name)) != expected_sha:
            raise ValueError(f"Installed NSB differs from selected wheel: {name}")
    nsb = importlib.import_module("nsb")
    imported = {
        Path(m.__file__).resolve()
        for name, m in list(sys.modules.items())
        if (name == "nsb" or name.startswith("nsb.")) and getattr(m, "__file__", None)
    }
    native = {p for p in paths if p.suffix in (".so", ".pyd")}
    if Path(nsb.__file__).resolve() not in paths or not native or not native.issubset(imported):
        raise ValueError("Shadowed NSB import or unexpected native extension")
    capabilities(nsb)
    return {
        "version": dist.version,
        "wheel_sha256": digest,
        "files": expected,
        "rust_siderust": str(getattr(nsb, "SIDERUST_VERSION", "unavailable")),
        "rust_tempoch": str(getattr(nsb, "TEMPOCH_VERSION", "unavailable")),
    }


def select_runtime():
    if RUNTIME == "developer":
        if not re.fullmatch("[0-9a-f]{40}", NSB_COMMIT):
            raise ValueError("Developer mode requires a full source commit assertion")
        if not re.fullmatch("[0-9a-f]{64}", NSB_WHEEL_SHA256):
            raise ValueError("Developer mode requires the expected wheel SHA-256")
        result = verify_wheel(NSB_WHEEL, NSB_WHEEL_SHA256, NSB_VERSION)
        if BUILD_RECEIPT:
            build = json.loads(Path(BUILD_RECEIPT).read_text())
            if (
                build.get("commit") != NSB_COMMIT
                or not build.get("source_clean")
                or build.get("wheel_sha256") != NSB_WHEEL_SHA256
                or build.get("version") != NSB_VERSION
                or build.get("installed_files") != wheel_files(NSB_WHEEL)
            ):
                raise ValueError("Optional build receipt differs from selected source/wheel")
            result["build_receipt"] = build
            result["build_receipt_sha256"] = sha256(BUILD_RECEIPT)
        return {
            **result,
            "mode": RUNTIME,
            "declared_source_commit": NSB_COMMIT,
            "source_commit_verification": "user-supplied build provenance",
        }
    if not re.fullmatch(r"[0-9]+\.[0-9]+\.[0-9]+(?:[a-z0-9.]+)?", NSB_VERSION):
        raise ValueError("Choose an exact PyPI version")
    registry_url = f"https://pypi.org/pypi/nsb-py/{NSB_VERSION}/json"
    with urlopen(registry_url, timeout=30) as response:
        registry = json.load(response)
    wheel_dir = CACHE / "wheels" / NSB_VERSION
    wheel_dir.mkdir(parents=True, exist_ok=True)
    # pip selects only compatible binary tags; source compilation is never implicit.
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "download",
            "--only-binary=:all:",
            "--no-deps",
            "--index-url",
            "https://pypi.org/simple",
            "--dest",
            str(wheel_dir),
            f"nsb-py=={NSB_VERSION}",
        ],
        check=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    wheels = list(wheel_dir.glob("*.whl"))
    if len(wheels) != 1:
        raise ValueError("Expected one compatible wheel; use an empty version cache")
    wheel = wheels[0]
    registered = next(f for f in registry["urls"] if f["filename"] == wheel.name)
    digest = registered["digests"]["sha256"]
    if sha256(wheel) != digest:
        raise ValueError("Downloaded/cached wheel differs from PyPI identity")
    if INSTALL_DEPENDENCIES:
        if "nsb" in sys.modules:
            raise RuntimeError("Restart kernel before selecting a published wheel")
        pip_install("--no-deps", "--force-reinstall", str(wheel))
    return {
        **verify_wheel(wheel, digest, NSB_VERSION),
        "mode": RUNTIME,
        "registry_url": registry_url,
    }


runtime_provenance = {}
if MODE == "observational":
    runtime_provenance["nsb"] = select_runtime()
    runtime_provenance["nsb2"] = verify_nsb2()
    from nsb2 import ASSETS_PATH

    skycalc_sha = sha256(ASSETS_PATH / "eso_skycalc_airglow_130sfu.dat")
    if skycalc_sha != "f3ac8d8260d3eafcf79ed80dc9cf2a04ecd852d30176ec44888764b491c4841e":
        raise ValueError("nsb2 SkyCalc differs from registered source")
    runtime_provenance["nsb2"]["skycalc_sha256"] = skycalc_sha
    display(Markdown(f"Verified NSB **{NSB_VERSION}**, nsb2 commit `{NSB2_COMMIT}`."))

In [ ]:
# v0.2.0 policy check: Automatic Airglow must match explicit Oxygen and exceed continuum.
import nsb
import siderust
from datetime import UTC, datetime

assert nsb.NsbModelConfig.generic_clear_sky().airglow_selection == "automatic"
observer = siderust.Observer(lon_deg=-70.4045, lat_deg=-24.6272, height_m=2635.0)
instant = datetime(2023, 9, 4, 1, 48, tzinfo=UTC)
target = siderust.Direction(ra_deg=0.0, dec_deg=-24.6272)
query = nsb.PointQuery(observer, instant, target, components=nsb.ComponentMask.AIRGLOW)
auto = float(nsb.NsbEvaluator(nsb.NsbModelConfig.generic_clear_sky()).integrate_band(query, 365.0, 650.0))
oxy = float(
    nsb.NsbEvaluator(
        nsb.NsbModelConfig.generic_clear_sky().with_airglow_model(nsb.AirglowModel.PARANAL_PALACE_OXYGEN_V1)
    ).integrate_band(query, 365.0, 650.0)
)
cont = float(
    nsb.NsbEvaluator(
        nsb.NsbModelConfig.generic_clear_sky().with_airglow_model(nsb.AirglowModel.PARANAL_PALACE_V1)
    ).integrate_band(query, 365.0, 650.0)
)
assert abs(auto - oxy) <= 1e-12 * max(1.0, abs(oxy)), (auto, oxy)
assert auto > cont, (auto, cont)
print(f"Automatic==Oxygen OK ({auto:.6f} ph cm^-2 ns^-1 sr^-1); continuum={cont:.6f}")


## 2. Registered observations and measurement

The [Patat (2008) FORS1 Night Sky Spectral Library](https://www.eso.org/~fpatat/science/skybright/)
is the sole observational dataset. The complete archive SHA-256 below is required
for download **and cache reuse**; its registered Wayback mirror contains identical
bytes. No substitute dataset is allowed. Unsafe TAR paths, links and duplicate
members are rejected; cached extracted FITS must match the verified archive.

Frozen sequential cuts require DARKFLG=0, 300V/FREE, Sun altitude < −18°,
Moon altitude < −5°, 1 ≤ airmass ≤ 1.5, |Galactic latitude| ≥ 20°, exposure ≥ 60 s,
and complete 365–650 nm coverage. Individual exclusion reasons are retained.
The exact ID/night/split checksum freezes **103 spectra / 22 nights**, with
**9 calibration spectra on five nights** and **94 evaluation spectra / 17 nights**.
Integer MJD defines the registered night; it is not regrouped by a new convention.

For sampled energy radiance $I_E$ in erg s⁻¹ cm⁻² Å⁻¹ arcsec⁻²,
$N=\int I_E(\lambda)\lambda/(hc)\,d\lambda$.
The operator integrates wavelength times the **piecewise-linear energy spectrum**
exactly (quadratic Simpson rule), inserting band edges. Å→cm, s→ns and
arcsec²→sr factors are explicit. Negative noisy spectral samples are retained;
nonpositive integrated observations fail. This is not a magnitude or count comparison.

In [ ]:
PATAT_SHA = "1f615fccb6dd464ab95b91f34adb4ba8ba3f4beebe97559204483c1da2c9965c"
PATAT_URLS = (
    "https://www.eso.org/~fpatat/science/skybright/DATA/skyspectra.tar",
    "https://web.archive.org/web/20160329190127id_/http://www.eso.org/~fpatat/science/skybright/DATA/skyspectra.tar",
)
SOLAR_SHA = "7240df111ea45f26057b89d3e9880a58856d7b6d040ad16247db41f5546dac96"
SOLAR_SLICE_SHA = "cd44cf6f2e162b763cfe3a216402f4a18d6b2ca88d05907496c2c2c738d225dc"
SOLAR_URL = (
    "https://www.ngdc.noaa.gov/stp/space-weather/solar-data/solar-features/"
    "solar-radio/noontime-flux/penticton/penticton_observed/listings/"
    "listing_drao_noontime-flux-observed_daily.txt"
)
CALIBRATION_NIGHTS = frozenset((51643, 51725, 51761, 51785, 52847))
# Exact SI constants expressed in cgs. 1 nm=10 A; 1 ns=1e-9 s.
HC_ERG_CM = 6.62607015e-27 * 2.99792458e10
ARCSEC2_PER_SR = (180 * 3600 / np.pi) ** 2
UNIT = "photons cm^-2 ns^-1 sr^-1"

COHORT_SHA = "3efdd3a0f3af19bfbc7615864f928b33fe22bcdbd6c67568fe317c059d44a1d6"


def acquire(path, expected, urls=(), *, offline=False):
    """Never replace an invalid cache; only atomically publish verified bytes."""
    path = Path(path)
    if path.is_symlink():
        raise ValueError("Input cache cannot be a symlink")
    if path.exists():
        if sha256(path) != expected:
            raise ValueError(f"Checksum mismatch: {path}; quarantine it explicitly")
        return path
    if offline:
        raise FileNotFoundError(f"Offline input unavailable: {path}")
    path.parent.mkdir(parents=True, exist_ok=True)
    failures = []
    for url in urls:
        print(f"Acquiring {url}", flush=True)
        try:
            with tempfile.NamedTemporaryFile(dir=path.parent) as out:
                with urlopen(url, timeout=45) as response:
                    while chunk := response.read(1024 * 1024):
                        out.write(chunk)
                        if out.tell() > 100_000_000:
                            raise ValueError("Input exceeds download size limit")
                out.flush()
                if sha256(out.name) != expected:
                    raise ValueError("Complete downloaded SHA-256 mismatch")
                # Hard link has exclusive-create semantics, so concurrent runs cannot overwrite.
                import os

                os.link(out.name, path)
            return path
        except Exception as error:
            failures.append(f"{url}: {type(error).__name__}: {error}")
    raise RuntimeError(
        "No verified input available; no substitute dataset used.\n" + "\n".join(failures)
    )


def extract(archive, destination, *, expected=PATAT_SHA):
    """Read regular flat FITS only; validate every TAR member before writing."""
    archive, destination = Path(archive), Path(destination)
    if sha256(archive) != expected:
        raise ValueError("Unverified archive")
    if any(p.is_symlink() for p in (destination, *destination.parents)):
        raise ValueError("Symlink in extraction destination")
    content = {}
    names = set()
    with tarfile.open(archive) as tar:
        for item in tar:
            name = Path(item.name)
            if (
                not item.isfile()
                or len(name.parts) != 1
                or name.is_absolute()
                or name.name in (".", "..")
                or item.name != name.name
                or item.name in names
                or item.size > 5_000_000
            ):
                raise ValueError(f"Unsafe/duplicate archive member: {item.name}")
            names.add(item.name)
            if item.name.startswith("fxSKY") and item.name.endswith(".fits"):
                content[item.name] = tar.extractfile(item).read()
    if not content:
        raise ValueError("No FITS in archive")
    inventory = {name: hashlib.sha256(data).hexdigest() for name, data in content.items()}
    if destination.exists():
        if {p.name for p in destination.iterdir()} != set(inventory):
            raise ValueError("Extracted inventory mismatch")
        for name, expected_digest in inventory.items():
            p = destination / name
            if p.is_symlink() or not p.is_file() or sha256(p) != expected_digest:
                raise ValueError(f"Extracted FITS changed: {name}")
    else:
        destination.parent.mkdir(parents=True, exist_ok=True)
        with tempfile.TemporaryDirectory(dir=destination.parent) as tmp:
            for name, data in content.items():
                (Path(tmp) / name).write_bytes(data)
            Path(tmp).rename(destination)
    return destination, inventory

In [ ]:
def linear_integral(x, y, lower, upper):
    """Trapezoid with interpolated boundaries; never extrapolate."""
    x, y = np.asarray(x, float), np.asarray(y, float)
    if (
        x.ndim != 1
        or x.size < 2
        or y.shape != x.shape
        or not np.isfinite(x).all()
        or not np.isfinite(y).all()
        or np.any(np.diff(x) <= 0)
        or not x[0] <= lower < upper <= x[-1]
    ):
        raise ValueError("Finite increasing full-coverage spectral samples required")
    nodes = np.r_[lower, x[(x > lower) & (x < upper)], upper]
    return float(np.trapezoid(np.interp(nodes, x, y), nodes))


def measured_radiance(wavelength_angstrom, energy, lower=365.0, upper=650.0):
    """Exact integral of lambda times piecewise-linear I_E, retaining noisy negatives.

    erg/s/cm2/A/arcsec2 -> photons/cm2/ns/sr. On each clipped interval,
    Simpson integration is exact for the quadratic product lambda*I_E.
    This differs from trapezoid integration of pre-converted photon samples.
    """
    wave = np.asarray(wavelength_angstrom, float)
    flux = np.asarray(energy, float)
    linear_integral(wave, flux, lower * 10, upper * 10)  # validate
    nodes = np.r_[lower * 10, wave[(wave > lower * 10) & (wave < upper * 10)], upper * 10]
    a, b = nodes[:-1], nodes[1:]
    mid = (a + b) / 2
    area = np.sum(
        (b - a)
        / 6
        * (
            a * np.interp(a, wave, flux)
            + 4 * mid * np.interp(mid, wave, flux)
            + b * np.interp(b, wave, flux)
        )
    )
    return float(area * 1e-8 / HC_ERG_CM * ARCSEC2_PER_SR * 1e-9)


def photon_density(wave_angstrom, energy):
    """Per-nm photon density; diagnostic only, broadband uses exact operator above."""
    return np.asarray(energy) * np.asarray(wave_angstrom) * 1e-8 / HC_ERG_CM * ARCSEC2_PER_SR * 1e-8


def exclusion(row):
    """Frozen sequential selection, including DARKFLG; boundaries are intentional."""
    cuts = (
        (row["dark"] != 0, "dark_flag"),
        (row["grism"] != "300V", "grism"),
        (row["filter"] != "FREE", "filter"),
        (row["sun_alt"] >= -18, "twilight"),
        (row["moon_alt"] >= -5, "moon"),
        (not 1 <= row["airmass"] <= 1.5, "airmass"),
        (abs(row["gal_lat"]) < 20, "galactic_latitude"),
        (row["exposure"] < 60, "exposure"),
        (row["wave"][0] > 3650 or row["wave"][-1] < 6500, "coverage"),
    )
    return next((reason for reject, reason in cuts if reject), None)


def load_cohort(directory, *, enforce_registration=True):
    rows, rejected = [], {}
    for path in sorted(Path(directory).glob("*.fits")):
        with fits.open(path) as hdus:
            hdr, flux = hdus[0].header, np.asarray(hdus[0].data, float)
            if hdr["BUNIT"].strip() != "erg/s/cm2/A/arcsec2":
                raise ValueError(f"Unexpected flux unit: {path.name}")
            wave = hdr["CRVAL1"] + (np.arange(hdr["NAXIS1"]) + 1 - hdr["CRPIX1"]) * hdr["CDELT1"]
            fields = {
                "mjd": "MJD-OBS",
                "ra": "RA",
                "dec": "DEC",
                "airmass": "AIRMASS",
                "sun_alt": "ALTSUN",
                "moon_alt": "ALTMOON",
                "gal_lat": "GALAT",
                "header_sfu": "SUNFLUX",
                "exposure": "ORIGEXP",
                "dark": "DARKFLG",
            }
            row = {key: float(hdr[value]) for key, value in fields.items()}
            if not np.isfinite(list(row.values())).all():
                raise ValueError(f"Nonfinite metadata: {path.name}")
            row.update(
                id=path.name,
                night=int(row["mjd"]),
                wave=wave,
                energy=flux,
                grism=hdr["GRISM"].strip(),
                filter=hdr["FILTER"].strip(),
            )
        reason = exclusion(row)
        if reason:
            rejected[row["id"]] = reason
            continue
        row["observed"] = measured_radiance(wave, flux)
        row["split"] = "calibration" if row["night"] in CALIBRATION_NIGHTS else "evaluation"
        if not np.isfinite(row["observed"]) or row["observed"] <= 0:
            raise ValueError("Nonpositive integrated observational radiance")
        rows.append(row)
    identities = "\n".join(f"{r['id']}:{r['night']}:{r['split']}" for r in rows).encode()
    identity_sha = hashlib.sha256(identities).hexdigest()
    if enforce_registration:
        if identity_sha != COHORT_SHA or len(rows) != 103:
            raise ValueError(
                "Selected observation identities/night partition differ from registration"
            )
    return rows, {
        "selected": len(rows),
        "nights": len({r["night"] for r in rows}),
        "excluded_ids": rejected,
        "identity_sha256": identity_sha,
    }

## 3. Solar inputs and cohort coverage

**Native** preserves each model's resolver: NSB's native solar history and nsb2's
Patat SUNFLUX rounded to 0.1 sfu. **Matched** uses independently observed NRC/Penticton
noontime F10.7 from the [NOAA mirror](https://www.ngdc.noaa.gov/stp/space-weather/solar-data/solar-features/solar-radio/noontime-flux/penticton/penticton_observed/listings/listing_drao_noontime-flux-observed_daily.txt),
with a complete centered 27-day UTC window (13 days each side). No interpolation,
imputation or fitting to residuals occurs. Only the pinned full file or registered
factual slice with the same parent identity is accepted. A growing NOAA file can
change; changed bytes fail verification and explicitly disable matched analysis.
A complete scientific campaign requires both scenarios.

PALACE's training domain is 67–166 sfu. We retain and identify extrapolations rather
than remove them. Observed/adjusted solar convention equivalence is not asserted.

In [ ]:
def solar_windows(path, rows):
    """Only checksum-pinned NOAA full source or frozen factual slice is accepted."""
    path = Path(path)
    digest = sha256(path)
    if digest == SOLAR_SLICE_SHA:
        payload = json.loads(path.read_text())
        if payload["source_sha256"] != SOLAR_SHA:
            raise ValueError("Solar source mismatch")
        daily = {date.fromisoformat(k): v for k, v in payload["daily"].items()}
    elif digest == SOLAR_SHA:
        daily = {}
        for line in path.read_text().splitlines():
            fields = line.split()
            if len(fields) == 2 and fields[1] != ".":
                daily[date(int(fields[0][:4]), int(fields[0][4:6]), int(fields[0][6:8]))] = float(
                    fields[1]
                )
    else:
        raise ValueError("Unverified solar data; matched scenario unavailable")
    values = {}
    for row in rows:
        center = Time(row["mjd"], format="mjd").to_datetime().date()
        days = [center + timedelta(days=i) for i in range(-13, 14)]
        if any(d not in daily for d in days):
            raise ValueError("Incomplete centered 27-day solar window")
        sample = np.array([daily[d] for d in days], float)
        if not np.isfinite(sample).all() or np.any(sample <= 0):
            raise ValueError("Invalid solar input")
        values[row["id"]] = float(np.mean(sample))
    return values

In [ ]:
SITE = dict(longitude=-70.4045, latitude=-24.6272, height=2635.0)
COMPONENTS = ("zodiacal", "airglow", "starlight", "moonlight")
MODELS = ("NSB Automatic", "NSB continuum", "nsb2")
DENSITY_UNIT = 1 / u.cm**2 / u.ns / u.sr / u.nm


def synthetic():
    """Small deterministic pipeline fixture; NEVER model/observational evidence."""
    rows = []
    predictions = {s: {m: [] for m in MODELS} for s in ("native", "matched")}
    wave = np.arange(3600.0, 6601.0, 5.0)
    for night in range(8):
        for exposure in range(3):
            i = len(rows)
            energy = 1.1e-17 * (1 + night * 0.05) * (1 + 0.1 * np.sin(wave / 100)) + 1e-16 * np.exp(
                -(((wave - 5577.34) / 10) ** 2)
            )
            row = dict(
                id=f"SYNTHETIC-{i:03}",
                night=60000 + night,
                split="calibration" if night < 2 else "evaluation",
                observed=measured_radiance(wave, energy),
                mjd=60000.0 + night + exposure * 0.01,
                ra=90.0,
                dec=-30.0,
                airmass=1.2,
                header_sfu=130.0,
                f27=90.0 + night * 20,
                wave=wave,
                energy=energy,
            )
            rows.append(row)
            for scenario in predictions:
                for j, model in enumerate(MODELS):
                    factor = (0.65, 0.85, 0.90)[j] + 0.09 * np.sin(i + j)
                    total = row["observed"] * factor
                    parts = dict(
                        zip(COMPONENTS, (total * 0.35, total * 0.5, total * 0.15, 0.0), strict=True)
                    )
                    predictions[scenario][model].append(
                        dict(id=row["id"], total=total, components=parts)
                    )
    bins = np.arange(550.0, 645.0)
    spectrum = dict(
        id=rows[0]["id"],
        air_wavelength_nm=(wave / 10).tolist(),
        observed_total_density=photon_density(wave, rows[0]["energy"]).tolist(),
        vacuum_bin_edges_nm=bins.tolist(),
        continuum=[0.0001] * 94,
        oxygen=[0.00015] * 94,
        scenario="SYNTHETIC fixture; not physical predictions",
    )
    return rows, predictions, spectrum


In [ ]:
provenance = dict(runtime_provenance)
availability, failures = {}, []
predictions, metrics, spectral = {}, {}, None
if MODE == "synthetic":
    rows, predictions, spectral = synthetic()
    selection = {"selected": len(rows), "identity_sha256": "SYNTHETIC"}
    availability = {s: "SYNTHETIC" for s in predictions}
else:
    from astropy.utils import data as astropy_data

    astropy_data.conf.allow_internet = not OFFLINE_DATA
    archive = acquire(CACHE / "skyspectra.tar", PATAT_SHA, PATAT_URLS, offline=OFFLINE_DATA)
    library, inventory = extract(archive, CACHE / "library")
    if len(inventory) != 1189:
        raise ValueError("Registered FITS product count changed")
    rows, selection = load_cohort(library)
    if (
        len({r["night"] for r in rows}),
        sum(r["split"] == "evaluation" for r in rows),
        len({r["night"] for r in rows if r["split"] == "evaluation"}),
    ) != (22, 94, 17):
        raise ValueError("Registered night/evaluation counts changed")
    provenance.update(
        dataset={"sha256": PATAT_SHA, "urls": PATAT_URLS, "fits_inventory": inventory},
        selection=selection,
    )
    availability["native"] = "available"
    try:
        solar = (
            Path(SOLAR_FILE)
            if SOLAR_FILE
            else acquire(
                CACHE / "solar-observed.txt", SOLAR_SHA, (SOLAR_URL,), offline=OFFLINE_DATA
            )
        )
        f27 = solar_windows(solar, rows)
        for row in rows:
            row["f27"] = f27[row["id"]]
        provenance["solar"] = {
            "input_sha256": sha256(solar),
            "source_sha256": SOLAR_SHA,
            "convention": "observed noontime; centered 27 complete UTC days",
            "outside_palace_training": sum(not 67 <= r["f27"] <= 166 for r in rows),
        }
        availability["matched"] = "available"
    except (OSError, ValueError, RuntimeError) as error:
        availability["matched"] = f"UNAVAILABLE: {error}"

evaluation = np.array([i for i, r in enumerate(rows) if r["split"] == "evaluation"])
y = np.array([r["observed"] for r in rows])[evaluation]
coverage = "| Partition | Spectra | Nights |\n|---|---:|---:|\n"
for split in ("calibration", "evaluation"):
    subset = [r for r in rows if r["split"] == split]
    coverage += f"| {split} | {len(subset)} | {len({r['night'] for r in subset})} |\n"
display(Markdown(coverage))
display(Markdown("**Scenario availability:** " + str(availability)))
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), layout="constrained")
for split in ("calibration", "evaluation"):
    subset = [r for r in rows if r["split"] == split]
    axes[0].scatter([r["mjd"] for r in subset], [r["observed"] for r in subset], label=split, s=18)
axes[0].set(xlabel="Observation time [MJD UTC]", ylabel=f"Observed [{UNIT}]")
axes[0].legend()
axes[1].hist([r["airmass"] for r in rows], bins=12)
axes[1].set(xlabel="Header airmass [dimensionless]", ylabel="Spectra")
fig.suptitle(f"{LABEL}: 365–650 nm coverage")
plt.show()

## 4. Physical models and predictions

NSB configurations use the actual public restricted-band evaluator.
**Automatic** (`generic_clear_sky`) resolves to `PARANAL_PALACE_OXYGEN_V1`.
**Explicit continuum** uses `PARANAL_PALACE_V1` and must not silently add Oxygen.
No replacement Oxygen approximation is introduced. Zodiacal, Airglow and Moon are integrated in-band;
Moon must be physically zero in the selected conditions.

**Research-only Starlight assumption:** the production integrated map has no
admitted restricted spectral product. We retain the registered Rieke solar-photon
fraction (365–650)/(300–650), multiplied by the integrated map. These conditional
research totals do **not** validate production spectral Starlight.

Pinned nsb2 uses public source radiances, including its faint Gaia DR3 G≥15
population, Leinert zodiacal source and combined ESO SkyCalc Airglow. Its native
zodiacal path has no added NSB extinction. Populations, propagation and Airglow
content differ even where component names agree. No telescope response is used.
The registered nsb2/Rieke integration uses trapezoids at in-band source-grid nodes;
it remains distinct from the exact observational measurement operator.

Matched nsb2 Airglow rescales its native source by the frozen scalar law
(0.2 + 0.00614 × measured F27)/(0.2 + 0.00614 × native SUNFLUX).
All 103 spectra must receive valid predictions from all three models in a scenario.


In [ ]:
def reference_integral(wave, density, lower, upper):
    """Frozen nsb2/Rieke protocol: trapezoid on in-band source-grid nodes.

    Retained to compare with original evidence (not a new integration rule).
    The exact measured-energy operator is separate and inserts band edges.
    """
    wave, density = np.asarray(wave, float), np.asarray(density, float)
    if (
        wave.shape != density.shape
        or wave.ndim != 1
        or not np.isfinite(density).all()
        or not np.isfinite(wave).all()
        or np.any(np.diff(wave) <= 0)
        or wave[0] > lower
        or wave[-1] < upper
    ):
        raise ValueError("Invalid source spectrum or incomplete coverage")
    mask = (wave >= lower) & (wave <= upper)
    if np.count_nonzero(mask) < 2:
        raise ValueError("Insufficient in-band source-grid nodes")
    return float(np.trapezoid(density[mask], wave[mask]))


def starlight_fraction():
    from nsb2.core.photometry import SolarSpectrumRieke2008

    wave, energy = SolarSpectrumRieke2008()
    photon = (energy * wave / (h * c)).to_value(1 / u.s / u.cm**2 / u.nm)
    wave = wave.to_value(u.nm)
    return reference_integral(wave, photon, 365, 650) / reference_integral(wave, photon, 300, 650)


def external_cache_inventory():
    """Record all Astropy data inputs, including nsb2's remote source assets."""
    return {url: sha256(path) for url, path in sorted(cache_contents().items())}

In [ ]:
class Models:
    def __init__(self):
        import siderust
        from nsb2.emitter.stars import from_gaia_dr3_map
        from nsb2.emitter.zodiacal import from_leinert1998

        import nsb

        iers.conf.auto_download = False
        self.nsb, self.siderust = nsb, siderust
        self.observer = siderust.Observer(
            lon_deg=SITE["longitude"], lat_deg=SITE["latitude"], height_m=SITE["height"]
        )
        self.site = EarthLocation.from_geodetic(
            SITE["longitude"] * u.deg, SITE["latitude"] * u.deg, SITE["height"] * u.m
        )
        print("Preparing nsb2 reference assets (Astropy cache; data downloads only)", flush=True)
        self.fraction = starlight_fraction()
        self.zodiacal = from_leinert1998()
        self.stars = from_gaia_dr3_map()
        self.air_sources = {}

    def query(self, row, mask):
        return self.nsb.PointQuery(
            self.observer,
            Time(row["mjd"], format="mjd", scale="utc").to_datetime(timezone=UTC),
            self.siderust.Direction(ra_deg=row["ra"], dec_deg=row["dec"]),
            components=mask,
        )

    def nsb_components(self, row, oxygen, sfu=None):
        nsb = self.nsb
        # Automatic == Oxygen for v0.2.0; continuum must be selected explicitly.
        if oxygen:
            config = nsb.NsbModelConfig.generic_clear_sky()  # Automatic → Oxygen
        else:
            config = nsb.NsbModelConfig.generic_clear_sky().with_airglow_model(
                nsb.AirglowModel.PARANAL_PALACE_V1
            )
        if sfu is not None:
            config = config.with_solar_radio_flux_sfu(float(sfu))
        evaluator = nsb.NsbEvaluator(config)
        # Includes Moon explicitly: API must establish physical zero, not a fabricated band.
        mask = nsb.ComponentMask.ZODIACAL | nsb.ComponentMask.AIRGLOW | nsb.ComponentMask.MOON
        band = evaluator.integrate_band_components(self.query(row, mask), 365.0, 650.0)
        result = {
            component.name: float(component.integrated_photons_cm2_ns_sr)
            for component in band.components
        }
        # Runtime uses "moon" rather than the report's "moonlight" label.
        result["moonlight"] = result.pop("moon")
        if result["moonlight"] != 0.0:
            raise ValueError("Registered moonless assumption violated")
        star = evaluator.evaluate(self.query(row, nsb.ComponentMask.STARLIGHT))
        result["starlight"] = float(star.components[0].integrated_photons_cm2_ns_sr) * self.fraction
        return {key: result[key] for key in COMPONENTS}

    def source_integral(self, source, row):
        frame = AltAz(
            obstime=Time(row["mjd"], format="mjd", scale="utc"),
            location=self.site,
            pressure=0 * u.hPa,
        )
        direction = SkyCoord(ra=[row["ra"]] * u.deg, dec=[row["dec"]] * u.deg)
        field, _ = source.query_direct(SimpleNamespace(origin=frame), direction, np.zeros(1))
        wave = field.spectral_grid.wvl.to_value(u.nm)
        raw = field.spectral_grid(field.spectral_data)
        try:
            density = (raw[:, 0] * field.weights[0, 0]).to_value(DENSITY_UNIT)
        except u.UnitConversionError:
            density = None
        if density is None or density.shape != wave.shape:
            # The public resolver handles the source's photometric normalization units.
            from nsb2.core.spectral import Bandpass

            band = Bandpass(np.array([365.0, 400.0, 550.0, 650.0]) * u.nm, np.ones(4))
            rate = field.resolve_spectra(band).integrate()
            return (
                float(
                    np.asarray(
                        rate.to_value(
                            1 / u.s / u.cm**2 / u.sr, equivalencies=u.dimensionless_angles()
                        )
                    ).reshape(-1)[0]
                )
                * 1e-9
            )
        return reference_integral(wave, density, 365, 650)

    def nsb2_components(self, row, sfu=None):
        from nsb2.emitter.airglow import from_eso_skycalc

        # Native nsb2 protocol uses SUNFLUX rounded to 0.1 sfu. Matched diagnostic
        # scales that exact source by its scalar law, with unrounded measured f27.
        native_flux = round(row["header_sfu"], 1)
        if native_flux not in self.air_sources:
            self.air_sources[native_flux] = from_eso_skycalc(90 * u.km, native_flux)
        air = self.source_integral(self.air_sources[native_flux], row)
        if sfu is not None:
            air *= (0.2 + 0.00614 * sfu) / (0.2 + 0.00614 * native_flux)
        return {
            "zodiacal": self.source_integral(self.zodiacal, row),
            "airglow": air,
            "starlight": self.source_integral(self.stars, row),
            "moonlight": 0.0,
        }

    def predict(self, row, model, sfu=None):
        parts = (
            self.nsb2_components(row, sfu)
            if model == "nsb2"
            else self.nsb_components(row, model == "NSB Automatic", sfu)
        )
        if (
            set(parts) != set(COMPONENTS)
            or not np.isfinite(list(parts.values())).all()
            or min(parts.values()) < 0
        ):
            raise ValueError("Invalid/incomplete component prediction")
        return {"id": row["id"], "total": float(sum(parts.values())), "components": parts}

    def spectral_example(self, row, sfu=None):
        """Actual restricted API bin integrals, never an invented resolved line profile."""
        nsb = self.nsb
        bins = np.arange(550.0, 645.0, 1.0)
        result = {
            "id": row["id"],
            "air_wavelength_nm": (row["wave"] / 10).tolist(),
            "observed_total_density": photon_density(row["wave"], row["energy"]).tolist(),
            "vacuum_bin_edges_nm": bins.tolist(),
            "scenario": "native" if sfu is None else "matched",
        }
        for name, model in (
            ("continuum", nsb.AirglowModel.PARANAL_PALACE_V1),
            ("automatic", None),
            ("oxygen", nsb.AirglowModel.PARANAL_PALACE_OXYGEN_V1),
        ):
            cfg = nsb.NsbModelConfig.generic_clear_sky()
            if model is not None:
                cfg = cfg.with_airglow_model(model)
            if sfu is not None:
                cfg = cfg.with_solar_radio_flux_sfu(sfu)
            ev = nsb.NsbEvaluator(cfg)
            result[name] = [
                float(
                    ev.integrate_band(
                        self.query(row, nsb.ComponentMask.AIRGLOW), float(a), float(b)
                    )
                )
                for a, b in zip(bins[:-1], bins[1:], strict=True)
            ]
        return result


In [ ]:
if MODE == "observational":
    models = Models()
    provenance["starlight_band_fraction"] = models.fraction
    for scenario, status in availability.items():
        if status != "available":
            continue
        predictions[scenario] = {m: [] for m in MODELS}
        for i, row in enumerate(rows):
            for model in MODELS:
                try:
                    predictions[scenario][model].append(
                        models.predict(row, model, row["f27"] if scenario == "matched" else None)
                    )
                except Exception as error:
                    failures.append(
                        dict(scenario=scenario, model=model, id=row["id"], reason=str(error))
                    )
            if (i + 1) % 25 == 0 or i + 1 == len(rows):
                print(f"{scenario}: {i + 1}/{len(rows)} spectra")
    spectral = models.spectral_example(rows[0], rows[0].get("f27"))
    provenance["astropy_input_cache"] = external_cache_inventory()
display(
    Markdown(
        f"**Rejected predictions: {len(failures)}** (IDs and reasons retained in result record)."
    )
)

## 5. Statistical evaluation

The baseline is the **median of the nine calibration spectra**, with no evaluation
nights used for fitting. Errors give every evaluation spectrum equal weight
(historically “exposure-weighted”, never duration-weighted). Residual = prediction −
observation. MAE, RMSE and bias carry radiance units; relative errors are dimensionless.

We resample **whole evaluation nights**, with replacement, 2,000 times using
NumPy `default_rng(182)`. Every model shares every draw, preserving pairing.
Percentile 95% intervals cover MAE and relative MAE, and every paired difference.
They condition on the fixed calibrated baseline and do not cover shared flux
calibration systematics. Missing, duplicate, extra, negative or nonfinite predictions
invalidate the entire scenario; we never score an opportunistic intersection.

The registered practical relative-MAE margin is **0.10**. A directional practical
claim requires the entire paired interval beyond that margin. Intervals crossing
zero do not demonstrate a difference. This rule grants no independent admission.

In [ ]:
def align(rows, predictions):
    """Exact identifier matching; reject duplicates, extras, missing/nonfinite predictions."""
    ids = [r["id"] for r in rows]
    if not ids or len(set(ids)) != len(ids):
        raise ValueError("Unique nonempty observation identities required")
    result = {}
    missing = {}
    for model, records in predictions.items():
        mapping = {r["id"]: r for r in records}
        missing[model] = sorted(set(ids) - set(mapping))
        if len(mapping) != len(records) or set(mapping) != set(ids):
            raise ValueError(f"{model}: cohort mismatch; missing={missing[model]}")
        values = np.array([mapping[key]["total"] for key in ids], float)
        if not np.isfinite(values).all() or np.any(values < 0):
            raise ValueError(f"{model}: invalid predictions")
        result[model] = values
    return result


def baseline(rows):
    train = [r for r in rows if r["split"] == "calibration"]
    test = [r for r in rows if r["split"] == "evaluation"]
    if not train or not test or set(r["night"] for r in train) & set(r["night"] for r in test):
        raise ValueError("Disjoint nonempty calibration and evaluation nights required")
    return float(np.median([r["observed"] for r in train]))


def errors(observed, predicted):
    y, p = np.asarray(observed, float), np.asarray(predicted, float)
    if (
        y.ndim != 1
        or y.size == 0
        or y.shape != p.shape
        or not np.isfinite(y).all()
        or not np.isfinite(p).all()
        or np.any(y <= 0)
    ):
        raise ValueError("Finite aligned vectors and positive observations required")
    d = p - y
    rel = d / y
    with np.errstate(over="raise", invalid="raise", divide="raise"):
        values = {
            "mean_signed_error": np.mean(d),
            "mae": np.mean(abs(d)),
            "rmse": np.linalg.norm(d / np.sqrt(len(d))),
            "median_absolute_error": np.median(abs(d)),
            "mean_absolute_relative_error": np.mean(abs(rel)),
            "median_absolute_relative_error": np.median(abs(rel)),
            "relative_bias": np.mean(rel),
            "relative_rmse": np.linalg.norm(rel / np.sqrt(len(rel))),
        }
    if not np.isfinite(list(values.values())).all():
        raise ValueError("Unstable metric values")
    return {k: float(v) for k, v in values.items()}


def scores(observed, predictions, nights, repetitions=2000, seed=182):
    y, nights = np.asarray(observed, float), np.asarray(nights)
    if nights.shape != y.shape or repetitions < 100 or not predictions:
        raise ValueError("Matching nights, models and >=100 repetitions required")
    groups = [np.flatnonzero(nights == n) for n in np.unique(nights)]
    if len(groups) < 3:
        raise ValueError("At least three independent nights required for intervals")
    predictions = {k: np.asarray(v, float) for k, v in predictions.items()}
    metrics = {
        k: {**errors(y, p), "n": len(y), "nights": len(groups), "missing": 0}
        for k, p in predictions.items()
    }
    rng = np.random.default_rng(seed)
    draws = {k: np.empty((repetitions, 2)) for k in predictions}
    for i in range(repetitions):
        indices = np.concatenate([groups[j] for j in rng.integers(len(groups), size=len(groups))])
        for key, p in predictions.items():
            delta = abs(p[indices] - y[indices])
            draws[key][i] = (np.mean(delta), np.mean(delta / y[indices]))
    for key in predictions:
        metrics[key]["mae_ci95"] = np.percentile(draws[key][:, 0], [2.5, 97.5]).tolist()
        metrics[key]["relative_mae_ci95"] = np.percentile(draws[key][:, 1], [2.5, 97.5]).tolist()
    paired = {}
    for a, b in combinations(predictions, 2):
        paired[f"{a}_minus_{b}"] = {
            "mae_difference": metrics[a]["mae"] - metrics[b]["mae"],
            "relative_mae_difference": metrics[a]["mean_absolute_relative_error"]
            - metrics[b]["mean_absolute_relative_error"],
            "mae_ci95": np.percentile(draws[a][:, 0] - draws[b][:, 0], [2.5, 97.5]).tolist(),
            "relative_mae_ci95": np.percentile(
                draws[a][:, 1] - draws[b][:, 1], [2.5, 97.5]
            ).tolist(),
        }
    return {
        "models": metrics,
        "paired": paired,
        "repetitions": repetitions,
        "seed": seed,
        "estimand": "equal weight per spectrum (historically called exposure-weighted); resample whole nights",
        "interval_scope": "sampling variability conditional on fixed calibration baseline; no calibration-systematic uncertainty",
    }

In [ ]:
baseline_value = baseline(rows)
provenance["baseline"] = {
    "value": baseline_value,
    "fit": "calibration-only spectrum median",
    "nights": sorted({r["night"] for r in rows if r["split"] == "calibration"}),
}
for scenario, records in predictions.items():
    try:
        vectors = align(rows, records)
        if set(vectors) != set(MODELS):
            raise ValueError("Required model missing")
    except ValueError as error:
        availability[scenario] = f"INVALID: {error}; no intersection scoring"
        continue
    vectors["baseline"] = np.full(len(rows), baseline_value)
    metrics[scenario] = scores(
        y,
        {k: v[evaluation] for k, v in vectors.items()},
        [rows[i]["night"] for i in evaluation],
        REPETITIONS,
        SEED,
    )
    table = (
        f"### {LABEL} — {scenario}\n\n"
        "| Model | Bias | MAE | RMSE | Median AE | Relative MAE [95% CI] | Relative RMSE |\n"
        "|---|---:|---:|---:|---:|---|---:|\n"
    )
    for name, m in metrics[scenario]["models"].items():
        lo, hi = m["relative_mae_ci95"]
        table += (
            f"| {name} | {m['mean_signed_error']:.5f} | {m['mae']:.5f} | {m['rmse']:.5f} | "
            f"{m['median_absolute_error']:.5f} | {m['mean_absolute_relative_error']:.5f} "
            f"[{lo:.5f}, {hi:.5f}] | {m['relative_rmse']:.5f} |\n"
        )
    display(
        Markdown(
            table
            + f"\nAbsolute columns: {UNIT}. Full metrics, relative bias and medians are in `metrics`."
        )
    )
    paired_table = "| Left minus right | Relative MAE difference | 95% paired CI | Interpretation |\n|---|---:|---|---|\n"
    for pair, p in metrics[scenario]["paired"].items():
        lo, hi = p["relative_mae_ci95"]
        decision = (
            "no demonstrated difference"
            if lo <= 0 <= hi
            else "left practically lower"
            if hi < -PRACTICAL_MARGIN
            else "right practically lower"
            if lo > PRACTICAL_MARGIN
            else "direction detected; practical margin not established"
        )
        p["interpretation"] = decision
        paired_table += (
            f"| {pair} | {p['relative_mae_difference']:.5f} | [{lo:.5f}, {hi:.5f}] | {decision} |\n"
        )
    display(Markdown(paired_table))

## 6. Predictions, errors and component contributions

Each scenario shows all 94 evaluation spectra in the frozen order. Full-sky
conditional research totals are compared with observed total radiance. Shared
residual bins retain underprediction and overprediction. Positive continuum-minus-
Automatic absolute-error differences indicate improvement; negative values indicate
**degradation**. Component bars disclose model contributions under their differing
source/propagation conventions. The baseline's uncertainty appears alongside models.


In [ ]:
figures = []


def show_figure(fig, name):
    fig.suptitle(f"{LABEL} — {name}", fontsize=11)
    figures.append((name, fig))
    plt.show()


for scenario, result in metrics.items():
    values = {
        m: np.array([r["total"] for r in predictions[scenario][m]])[evaluation] for m in MODELS
    }
    fig, axes = plt.subplots(2, 3, figsize=(16, 9), layout="constrained")
    for model, p in values.items():
        axes[0, 0].scatter(y, p, s=15, alpha=0.6, label=model)
    high = max(y.max(), *(p.max() for p in values.values())) * 1.05
    axes[0, 0].plot([0, high], [0, high], color="black", lw=1)
    axes[0, 0].set(xlabel=f"Observed [{UNIT}]", ylabel=f"Predicted [{UNIT}]")
    axes[0, 0].legend(fontsize=8)
    residuals = {m: (p - y) / y for m, p in values.items()}
    bins = np.linspace(
        min(v.min() for v in residuals.values()), max(v.max() for v in residuals.values()), 21
    )
    for model, v in residuals.items():
        axes[0, 1].hist(v, bins=bins, histtype="step", label=model)
    axes[0, 1].axvline(0, color="black", lw=1)
    axes[0, 1].set(xlabel="Signed relative residual [dimensionless]", ylabel="Spectra")
    for j, (_model, m) in enumerate(result["models"].items()):
        axes[0, 2].plot(m["relative_mae_ci95"], [j, j], color=f"C{j}")
        axes[0, 2].scatter(m["mean_absolute_relative_error"], j, color=f"C{j}")
    axes[0, 2].set(
        yticks=range(4),
        yticklabels=list(result["models"]),
        xlabel="Relative MAE [dimensionless]; 95% night CI",
    )
    delta = abs(residuals["NSB continuum"]) - abs(residuals["NSB Automatic"])
    axes[1, 0].bar(range(len(y)), delta, color=np.where(delta >= 0, "tab:blue", "tab:red"), width=1)
    axes[1, 0].axhline(0, color="black", lw=1)
    axes[1, 0].set(
        xlabel="Evaluation spectrum index (frozen ID order)",
        ylabel="Relative AE(continuum) − AE(Automatic/Oxygen)",
    )
    bottom = np.zeros(3)
    for part in COMPONENTS:
        v = [
            np.mean([predictions[scenario][m][i]["components"][part] for i in evaluation])
            for m in MODELS
        ]
        axes[1, 1].bar(MODELS, v, bottom=bottom, label=part)
        bottom += v
    axes[1, 1].set(ylabel=f"Mean component [{UNIT}]")
    axes[1, 1].legend(fontsize=8)
    nights = np.array([rows[i]["night"] for i in evaluation])
    for model, residual in residuals.items():
        axes[1, 2].plot(
            np.unique(nights),
            [np.mean(residual[nights == n]) for n in np.unique(nights)],
            marker="o",
            markersize=3,
            label=model,
        )
    axes[1, 2].axhline(0, color="black", lw=1)
    axes[1, 2].set(xlabel="Independent night [integer MJD]", ylabel="Mean signed relative residual")
    axes[1, 2].legend(fontsize=8)
    show_figure(fig, f"{scenario}: 365–650 nm evaluation")


## 7. Spectral features and solar sensitivity

The spectral panels deliberately separate **observed total radiance on air
wavelengths** from **NSB Airglow-only vacuum-bin integrals divided by the 1 nm bin
width**. These are different observables, shown as a component diagnostic, with
no total-versus-Airglow goodness-of-fit score. The NSB API supplies real Oxygen
features; these are not invented line profiles or instrument-convolved spectra.

Solar panels retain native and matched inputs separately. The shaded 67–166 sfu
region is PALACE's training range, not a selection cut. Correlation of errors with
F10.7 alone does not identify a solar causal effect: viewing conditions covary.

In [ ]:
if spectral:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), layout="constrained")
    edges = np.array(spectral["vacuum_bin_edges_nm"])
    for ax, limits in zip(axes[:2], ((552, 563), (625, 642)), strict=True):
        ax.plot(
            spectral["air_wavelength_nm"],
            spectral["observed_total_density"],
            color="black",
            label="Observed TOTAL; air wavelength",
        )
        for name in ("continuum", "oxygen"):
            ax.stairs(
                np.array(spectral[name]) / np.diff(edges),
                edges,
                label=f"NSB AIRGLOW {name}; vacuum bins",
            )
        ax.set(xlim=limits, xlabel="Wavelength [nm]", ylabel=f"Radiance density [{UNIT} nm^-1]")
    axes[0].legend(fontsize=7)
    if "matched" in metrics:
        f27 = np.array([rows[i]["f27"] for i in evaluation])
        axes[2].axvspan(67, 166, alpha=0.12, color="green", label="PALACE training domain")
        for model in MODELS:
            native = np.array([r["total"] for r in predictions["native"][model]])[evaluation]
            matched = np.array([r["total"] for r in predictions["matched"][model]])[evaluation]
            axes[2].scatter(f27, (abs(native - y) - abs(matched - y)) / y, s=12, label=model)
        axes[2].axhline(0, color="black", lw=1)
        axes[2].set(
            xlabel="Observed centered 27-day F10.7 [sfu]",
            ylabel="Relative AE(native) − AE(matched)",
        )
        axes[2].legend(fontsize=7)
    else:
        axes[2].text(0.1, 0.5, "Matched solar scenario unavailable", transform=axes[2].transAxes)
    show_figure(fig, f"Spectral component diagnostic ({spectral['id']}); solar-input sensitivity")
    if "solar" in provenance:
        display(
            Markdown(
                f"**{provenance['solar']['outside_palace_training']}/103 observations outside PALACE's solar training range.**"
            )
        )

## 8. Save a reproducibility record; compare releases

A run exports one JSON record plus a detached SHA-256, optionally PNG figures.
It includes observations, every component prediction and residual-reconstructing
total, metrics, paired intervals, exclusions, runtime/wheel identities, all installed
dependency versions and external source-cache hashes. No export is needed to run
or share this notebook. Preserve the record and its digest together, and retain a
trusted digest separately if archival authenticity matters. Checksums detect mutation;
they are not signatures.

For version A, retain the printed result JSON path. Start a fresh kernel with version
B selected and put A's path in `PREVIOUS_RUN`. Both runs must be observational and
complete. Comparison checks the entire frozen protocol, observed values and split,
solar inputs, reference sources, source-cache hashes, statistical definitions,
Starlight proxy and required provider versions. NSB wheel/version changes are
allowed; unrelated dependency/source changes are rejected to prevent confounding.
A protocol change requires a new experiment, not a comparable release score.
Normalized scientific definitions actually submitted to the kernel are hashed;
algorithm edits are therefore incompatible even if someone forgets to rename the protocol.

The plot shows B−A relative MAE (positive = regression), with paired night-bootstrap
intervals computed from aligned per-observation predictions across runs. The
practical threshold remains 0.10. Comparing a record with itself is a useful
zero-difference engineering check, not evidence about two releases.

In [ ]:
def canonical(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"), allow_nan=False)


def record_digest(value):
    return hashlib.sha256(canonical(value).encode()).hexdigest()


def scientific_signature(history):
    """Hash normalized scientific definitions actually submitted to this kernel."""
    required = set(
        "acquire extract linear_integral measured_radiance photon_density exclusion load_cohort solar_windows reference_integral starlight_fraction Models align baseline errors scores".split()
    )
    definitions = {}
    for source in history:
        try:
            tree = ast.parse(source)
        except SyntaxError:
            continue  # unrelated interactive magics are not scientific definitions
        for node in tree.body:
            if isinstance(node, (ast.FunctionDef, ast.ClassDef)) and node.name in required:
                definitions[node.name] = ast.dump(node, include_attributes=False)
    if set(definitions) != required:
        raise ValueError("Run all scientific definition cells before recording results")
    return record_digest(definitions)


def make_record():
    compact = [{k: v for k, v in r.items() if k not in ("wave", "energy")} for r in rows]
    dependencies = {d.metadata["Name"]: d.version for d in metadata.distributions()}
    protocol = {
        "name": "patat-fors1-365-650-v1",
        "scientific_code_sha256": scientific_signature(
            get_ipython().history_manager.input_hist_raw
        ),
        "dataset_sha256": PATAT_SHA,
        "cohort_sha256": selection["identity_sha256"],
        "band_nm": [365, 650],
        "unit": UNIT,
        "measurement": "exact integral of wavelength times piecewise-linear energy; air grid",
        "reference_integration": "in-band grid-node trapezoid; public template resolver",
        "solar": provenance.get("solar", {}),
        "scenarios": sorted(metrics),
        "calibration_nights": sorted(CALIBRATION_NIGHTS),
        "baseline": "calibration spectrum median; fixed during resampling",
        "statistical_definition": "equal-spectrum MAE/RMSE/signed bias and medians; whole-night percentile bootstrap",
        "repetitions": REPETITIONS,
        "seed": SEED,
        "practical_margin": PRACTICAL_MARGIN,
        "invalid_predictions": "whole scenario invalid; no intersection scoring",
        "models": list(MODELS),
        "site": SITE,
        "starlight": {
            "assumption": "research-only Rieke photon fraction",
            "fraction": provenance.get("starlight_band_fraction"),
        },
        "reference": provenance.get("nsb2", {}),
        "external_sources": provenance.get("astropy_input_cache", {}),
        "dependencies": {
            k: metadata.version(k)
            for k in (
                list(DEPENDENCIES)
                + ["astropy-iers-data", "scipy", "healpy", "scikit-learn", "dust_extinction"]
                if MODE == "observational"
                else ["numpy", "astropy", "matplotlib"]
            )
        },
    }
    return {
        "schema": 1,
        "classification": LABEL,
        "mode": MODE,
        "complete": MODE == "observational"
        and set(metrics) == {"native", "matched"}
        and not failures,
        "independent_validation": False,
        "created_utc": datetime.now(UTC).isoformat(),
        "protocol": protocol,
        "provenance": provenance,
        "observations": compact,
        "predictions": predictions,
        "metrics": metrics,
        "availability": availability,
        "failures": failures,
        "spectral": spectral,
        "environment": {
            "python": sys.version,
            "platform": platform.platform(),
            "packages": dependencies,
        },
    }


def save_record(record, directory):
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    name = datetime.now(UTC).strftime("run-%Y%m%dT%H%M%S-") + record_digest(record)[:12]
    path = directory / (name + ".json")
    with path.open("x") as stream:
        stream.write(canonical(record) + "\n")
    path.with_suffix(".sha256").write_text(sha256(path) + "\n")
    return path


def load_record(path):
    path = Path(path)
    if sha256(path) != path.with_suffix(".sha256").read_text().strip():
        raise ValueError("Result record SHA-256 mismatch")
    record = json.loads(path.read_text())
    if record.get("schema") != 1:
        raise ValueError("Unsupported result record schema")
    return record


def compare_records(previous, current):
    for record in (previous, current):
        if (
            record.get("schema") != 1
            or record.get("mode") != "observational"
            or not record.get("complete")
        ):
            raise ValueError("Only complete observational result records are comparable")
    if canonical(previous["protocol"]) != canonical(current["protocol"]):
        raise ValueError(
            "Incompatible protocol, cohort, solar scenario, statistics or source identities"
        )
    # Stronger than IDs alone: verify actual measured radiances, geometry, dates and solar values.
    if canonical(previous["observations"]) != canonical(current["observations"]):
        raise ValueError("Incompatible observed values or selection")
    selected = [r for r in current["observations"] if r["split"] == "evaluation"]
    indices = [i for i, r in enumerate(current["observations"]) if r["split"] == "evaluation"]
    observed = np.array([r["observed"] for r in selected])
    differences = {}
    for scenario in current["protocol"]["scenarios"]:
        old = align(previous["observations"], previous["predictions"][scenario])
        new = align(current["observations"], current["predictions"][scenario])
        if set(old) != set(MODELS) or set(new) != set(MODELS):
            raise ValueError("Missing required model in version comparison")
        differences[scenario] = {}
        for model in MODELS:
            result = scores(
                observed,
                {"current": new[model][indices], "previous": old[model][indices]},
                [r["night"] for r in selected],
                REPETITIONS,
                SEED,
            )
            differences[scenario][model] = result["paired"]["current_minus_previous"]
    return differences


notebook_file = Path("NSB_Validation.ipynb")
if notebook_file.is_file():
    notebook_payload = json.loads(notebook_file.read_text())
    sources = [
        "".join(cell["source"]) for cell in notebook_payload["cells"] if cell["cell_type"] == "code"
    ]
    provenance["notebook"] = {
        "file_sha256": sha256(notebook_file),
        "on_disk_code_sha256": record_digest(sources),
        "scope": "on-disk file; save editor before Run All",
    }
else:
    provenance["notebook"] = {"scope": "not found in working directory; save notebook with results"}
record = make_record()
result_path = None
if OUTPUT:
    result_path = save_record(record, OUTPUT)
    for i, (_name, fig) in enumerate(figures):
        fig.savefig(result_path.with_name(result_path.stem + f"-figure-{i + 1}.png"), dpi=140)
    display(Markdown(f"Saved **{result_path.resolve()}** and detached SHA-256."))
if PREVIOUS_RUN:
    comparison = compare_records(load_record(PREVIOUS_RUN), record)
    fig, axes = plt.subplots(
        1, len(comparison), figsize=(12, 4), squeeze=False, layout="constrained"
    )
    for ax, (scenario, values) in zip(axes[0], comparison.items(), strict=True):
        for j, (_model, diff) in enumerate(values.items()):
            ax.plot(diff["relative_mae_ci95"], [j, j], color=f"C{j}")
            ax.scatter(diff["relative_mae_difference"], j, color=f"C{j}")
        ax.axvline(0, color="black", lw=1)
        ax.axvline(PRACTICAL_MARGIN, color="red", ls=":")
        ax.axvline(-PRACTICAL_MARGIN, color="red", ls=":")
        ax.set(
            yticks=range(3),
            yticklabels=list(values),
            title=scenario,
            xlabel="Current − previous relative MAE; 95% paired CI",
        )
    show_figure(fig, "Version comparison: positive = regression")

## 9. Interpretation, limitations and provenance

Computed values and figures above come from this kernel. The following are
**historical context only**, never substituted for computation: matched relative
MAE was approximately continuum **0.35027**, Automatic/Oxygen **0.16573**, nsb2 **0.17425**,
and calibration baseline **0.15245**. Historical Oxygen−nsb2 relative-MAE difference
was −0.00852 with paired 95% CI [−0.05709, 0.05576]: **superiority was not established**.
The baseline had lower relative MAE. Original benchmark revision
`1b2556727205694c7765ef0d254c5d1b301582ec`, Oxygen numerical revision
`168e73dae89060423eebc344871b0084d217c305`, and later merge
`57584ee298f25f8af7a6f169d71e87f8a59eea9a` are distinct identities.
A new version agreeing numerically is not a byte-identical historical replay.

Inspect paired intervals, night concentration, individual degradations and solar
extrapolation before interpreting means. Missing non-oxygen emissions and molecular
absorption, different stellar populations, the research SED proxy and differing
transport conventions limit physical equivalence. FORS flux calibration has shared
few-percent uncertainty; no per-spectrum statistical uncertainty is supplied.
Bootstrap intervals quantify night sampling, not those systematics. This restricted
single-site dark-time cohort cannot establish generalization or site calibration.
An independent post-development cohort with compatible diffuse calibration is
needed for stronger scientific claims. No automatic/default model is changed.

The reproducibility record retains the exact input/cohort/reference identities,
per-observation outputs and environment. The notebook below records a hash of its
on-disk executable cells when found; unsaved editor changes are explicitly outside
that file identity. Save before Run All for an archival run. No repository identity
is inferred. Future methodological edits must change the protocol identifier and
compatibility definition deliberately.

Sources to cite: [Patat (2008), A&A 481, 575](https://www.aanda.org/articles/aa/pdf/2008/14/aa9279-07.pdf);
[Roellinghoff, Spencer & Funk (2025), A&A 698, A212](https://www.aanda.org/articles/aa/pdf/2025/06/aa54532-25.pdf);
[nsb2 source and acknowledgements](https://gerritro.github.io/nsb2/acknowledgements.html);
[NSB Oxygen development evidence](https://github.com/VPRamon/NSB/blob/57584ee298f25f8af7a6f169d71e87f8a59eea9a/docs/nsb_components/airglow/validation/oxygen-and-calibration-v1.md).
The stellar path uses ESA Gaia data processed by DPAC and HEALPix/healpy; retain
original source credits and data terms. No private validation source is used.


In [ ]:
display(Markdown(f"**{LABEL}** — complete observational campaign: **{record['complete']}**"))
display(
    Markdown(
        "Runtime and data provenance are retained in `record['provenance']`; "
        "all installed package versions in `record['environment']`."
    )
)
if MODE == "observational" and not record["complete"]:
    raise RuntimeError(
        "Incomplete scientific campaign: inspect availability/failures in the saved record"
    )